# 00 — Verificação da base de modelagem

Checagens de sanidade na camada Gold antes da modelagem:

- **Query A**: grão, ano de cada fonte do Censo Escolar, joins que falharam, nulos e distribuição do alvo na base enriquecida.
- **Query B**: cobertura por ano e presença de metas em `metas_resultados_municipio`.
- **Query C**: cobertura do `indicador_2023` para os municípios da base enriquecida (viabilidade do enquadramento com histórico).

In [8]:
import pandas as pd

from src.data.bq_client import DATASET_GOLD, PROJECT_ID, get_client, run_query

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

client = get_client()

T_ENRIQ = f"`{PROJECT_ID}.{DATASET_GOLD}.metas_resultados_municipio_enriquecido_2024`"
T_METAS = f"`{PROJECT_ID}.{DATASET_GOLD}.metas_resultados_municipio`"
T_EVOL = f"`{PROJECT_ID}.{DATASET_GOLD}.evolucao_indicador_municipio`"

print("Conectado a:", client.project)

Conectado a: grupo-pos-fase-2-alfabetizacao


## Query A — Sanidade da base enriquecida

O resultado é uma única linha com muitas métricas; exibido transposto (uma métrica por linha) para leitura.

In [9]:
sql_a = f"""
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT id_municipio) AS municipios,
  STRING_AGG(DISTINCT CAST(ano AS STRING)) AS anos_alvo,
  STRING_AGG(DISTINCT CAST(ano_features AS STRING)) AS anos_features,
  STRING_AGG(DISTINCT CAST(CAST(mat_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_mat,
  STRING_AGG(DISTINCT CAST(CAST(had_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_had,
  STRING_AGG(DISTINCT CAST(CAST(afd_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_afd,
  STRING_AGG(DISTINCT CAST(CAST(ird_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_ird,
  STRING_AGG(DISTINCT CAST(CAST(tdi_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_tdi,
  STRING_AGG(DISTINCT CAST(CAST(tre_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_tre,
  STRING_AGG(DISTINCT CAST(CAST(dsu_ano_nu_ano_censo AS INT64) AS STRING)) AS ano_dsu,
  COUNTIF(mat_codigo_do_municipio_co_municipio IS NULL) AS sem_match_mat,
  COUNTIF(had_codigo_do_municipio_co_municipio IS NULL) AS sem_match_had,
  COUNTIF(afd_codigo_do_municipio_co_municipio IS NULL) AS sem_match_afd,
  COUNTIF(ird_codigo_do_municipio_co_municipio IS NULL) AS sem_match_ird,
  COUNTIF(tdi_codigo_do_municipio_co_municipio IS NULL) AS sem_match_tdi,
  COUNTIF(tre_codigo_do_municipio_co_municipio IS NULL) AS sem_match_tre,
  COUNTIF(dsu_codigo_do_municipio_co_municipio IS NULL) AS sem_match_dsu,
  COUNTIF(CAST(mat_codigo_do_municipio_co_municipio AS INT64) != id_municipio) AS chave_divergente,
  COUNTIF(ibge_pib_per_capita IS NULL) AS pib_nulo,
  COUNTIF(atingiu_meta) AS atingiram_meta,
  ROUND(AVG(indicador_alfabetizacao), 2) AS media_indicador,
  ROUND(MIN(indicador_alfabetizacao), 2) AS min_indicador,
  ROUND(MAX(indicador_alfabetizacao), 2) AS max_indicador
FROM {T_ENRIQ}
"""

df_a = run_query(sql_a, client=client)
df_a.T.rename(columns={0: "valor"})

,valor
linhas,5232
municipios,5232
anos_alvo,2024
anos_features,2023
ano_mat,2023
ano_had,2023
ano_afd,2023
ano_ird,2023
ano_tdi,2023
ano_tre,2023


## Query B — Cobertura por ano e metas

In [10]:
sql_b = f"""
SELECT
  ano,
  COUNT(*) AS linhas,
  COUNT(DISTINCT id_municipio) AS municipios,
  COUNTIF(meta_alfabetizacao IS NULL) AS meta_nula,
  COUNTIF(atingiu_meta) AS atingiram_meta
FROM {T_METAS}
GROUP BY ano
ORDER BY ano
"""

df_b = run_query(sql_b, client=client)
df_b

,ano,linhas,municipios,meta_nula,atingiram_meta
0,2024,5232,5232,0,2791


## Query C — Cobertura do indicador 2023 na base enriquecida

`id_municipio` é INT64 na base enriquecida e STRING em `evolucao_indicador_municipio`: o join converte com CAST (seguro, pois código IBGE nunca começa com zero).

In [11]:
sql_c = f"""
SELECT
  COUNT(*) AS municipios_enriquecida,
  COUNTIF(e.id_municipio IS NULL) AS sem_match_evolucao,
  COUNTIF(e.indicador_2023 IS NOT NULL) AS com_indicador_2023,
  COUNTIF(e.id_municipio IS NOT NULL AND e.indicador_2023 IS NULL) AS match_sem_2023
FROM {T_ENRIQ} AS m
LEFT JOIN {T_EVOL} AS e
  ON CAST(m.id_municipio AS STRING) = e.id_municipio
"""

df_c = run_query(sql_c, client=client)
df_c.T.rename(columns={0: "valor"})

,valor
municipios_enriquecida,5232
sem_match_evolucao,621
com_indicador_2023,4611
match_sem_2023,0


## Query D — Distribuição por UF dos municípios sem indicador 2023

Verifica se a ausência de histórico é aleatória ou concentrada em poucas UFs (ausência informativa).

In [12]:
sql_d = f"""
SELECT
  m.sigla_uf,
  COUNT(*) AS municipios,
  COUNTIF(e.id_municipio IS NULL) AS sem_2023,
  ROUND(100 * COUNTIF(e.id_municipio IS NULL) / COUNT(*), 1) AS pct_sem_2023
FROM {T_ENRIQ} AS m
LEFT JOIN {T_EVOL} AS e
  ON CAST(m.id_municipio AS STRING) = e.id_municipio
GROUP BY m.sigla_uf
HAVING COUNTIF(e.id_municipio IS NULL) > 0
ORDER BY sem_2023 DESC
"""

df_d = run_query(sql_d, client=client)
print(df_d.to_string(index=False))

sigla_uf  municipios  sem_2023  pct_sem_2023
      SP         621       621         100.0


## Query E — Estrutura da Silver (`consolidado_municipio`)

Verifica: (1) duplicação de linhas por município–ano–rede; (2) valores de `rede` por ano (hierarquia agregado × atômico); (3) presença de SP em 2023 — se existe na Silver, a ausência na Gold vem da construção da Gold, não da fonte.

In [13]:
T_SILVER = f"`{PROJECT_ID}.silver.consolidado_municipio`"

sql_e = f"""
SELECT
  ano,
  rede,
  COUNT(*) AS linhas,
  COUNT(DISTINCT id_municipio) AS municipios,
  COUNT(DISTINCT CONCAT(id_municipio, '-', CAST(ano AS STRING), '-', rede)) AS chaves_unicas,
  COUNTIF(sigla_uf = 'SP') AS linhas_sp,
  COUNT(DISTINCT IF(sigla_uf = 'SP', id_municipio, NULL)) AS municipios_sp
FROM {T_SILVER}
GROUP BY ano, rede
ORDER BY ano, rede
"""

df_e = run_query(sql_e, client=client)
print(df_e.to_string(index=False))

 ano rede  linhas  municipios  chaves_unicas  linhas_sp  municipios_sp
2023    2   52087        1063           1063          0              0
2023    3 1609209        4825           4825          0              0
2024    2   53326        1090           1090       4557             93
2024    3 1833832        5452           5452     214473            626
2024    4       1           1              1          0              0


## Query F — Qual rede o indicador da Gold representa

Compara, por rede, o `indicador_alfabetizacao` da Gold com o da Silver (2024). Também verifica se as cópias duplicadas da Silver têm valores idênticos de indicador e `qtd_alunos` (premissa para deduplicar com segurança).

In [14]:
sql_f = f"""
WITH silver_dedup AS (
  SELECT
    id_municipio,
    rede,
    ANY_VALUE(indicador_alfabetizacao) AS ind_silver,
    COUNT(DISTINCT indicador_alfabetizacao) AS n_valores_indicador,
    COUNT(DISTINCT qtd_alunos) AS n_valores_qtd
  FROM {T_SILVER}
  WHERE ano = 2024 AND rede IN ('2', '3')
  GROUP BY id_municipio, rede
)
SELECT
  s.rede,
  COUNT(*) AS municipios_comparados,
  COUNTIF(ABS(s.ind_silver - g.indicador_alfabetizacao) < 0.01) AS valores_iguais,
  ROUND(AVG(ABS(s.ind_silver - g.indicador_alfabetizacao)), 2) AS dif_media_abs,
  MAX(s.n_valores_indicador) AS max_valores_indicador_por_chave,
  MAX(s.n_valores_qtd) AS max_valores_qtd_por_chave
FROM {T_ENRIQ} AS g
JOIN silver_dedup AS s
  ON CAST(g.id_municipio AS STRING) = s.id_municipio
GROUP BY s.rede
ORDER BY s.rede
"""

df_f = run_query(sql_f, client=client)
print(df_f.to_string(index=False))

rede  municipios_comparados  valores_iguais  dif_media_abs  max_valores_indicador_por_chave  max_valores_qtd_por_chave
   2                    925               2          14.31                                1                          1
   3                   5232            5232           0.00                                1                          1


## Conclusões

**Grão e cobertura**
- Base enriquecida: 5.232 municípios, uma linha por município (~94% dos 5.570 do país).
- O alvo (`indicador_alfabetizacao`, 2024) é o Indicador Criança Alfabetizada da **rede municipal** (rede 3): coincide com a Silver em 5.232 de 5.232 municípios. A rede estadual, presente em 925 desses municípios, não compõe o alvo.
- Escala de 0 a 100; média por município de 63,3 (não ponderada). O mínimo de 6,25 (= 1/16) indica denominadores pequenos → modelagem deve ponderar pelo número de alunos.

**Precedência temporal**
- As 7 fontes do Censo Escolar referem-se a 2023; o alvo, a 2024. As features precedem o alvo, o que torna legítimas inclusive as taxas de rendimento (`tre_`).

**Qualidade do enriquecimento**
- Chave do join consistente (0 divergências); 1 município sem IRD; PIB per capita sem nulos.

**Histórico (2023)**
- `metas_resultados_municipio` contém apenas 2024.
- `indicador_2023` existe para 4.611 municípios (88,1%). Os 621 ausentes são todos de SP, e SP não tem dados de 2023 já na Silver: a ausência vem da fonte, não da construção da Gold. Qualquer modelagem com histórico fica restrita a "Brasil exceto SP".

**Silver (`consolidado_municipio`)**
- Linhas multiplicadas por join (~49 cópias por chave na rede estadual, ~335 na municipal). As cópias têm valores idênticos de indicador e `qtd_alunos`, então a deduplicação é segura.
- Contém apenas as redes 2 e 3, e uma linha da rede 4 (a descartar).
- Fonte do número de alunos para ponderação: Silver, rede 3, ano 2024, deduplicada.

**Pendência**
- Confirmar qual linha de dependência/localização das planilhas do INEP foi usada no enriquecimento (o ideal, para alinhar ao alvo, é a dependência municipal).